# Mock Interview Practice — Day 26
**FAANG / Top Product Company Level — Week 4, Day 5** · Wed Nov 18, 2026

*Days 22–25 built the FAANG tier: sketches, streaming semantics, lakehouse internals, storage algorithms, release engineering. Day 26 is the synthesis stretch: statistics that rank fairly, concurrency and precision traps in pure Python, the built-in sessionization Day 19 deferred, incremental maintenance proven with a correctness harness, and the finale triad — dead-man's-switch monitoring, blameless incident command, and a multi-system 3 AM capstone. All 18 topics are fresh vs days 1–25 (verified by a full-title sweep of all 25 notebooks).*

**18 questions · 8 categories · 9 [FAANG] + 9 [HARD]**

| # | Category | Questions | Topics |
|---|----------|-----------|--------|
| 1 | SQL Advanced | Q1–Q2 | Wilson lower-bound ranking · Benford's-law fraud screen |
| 2 | Python — Loops/Functions/Lists/Dicts | Q3–Q4 | bounded-buffer producer-consumer (threads) · floating-point error & Kahan summation |
| 3 | PySpark Coding | Q5–Q6 | built-in `session_window` event-time sessionization · CDF-driven incremental rollup maintenance |
| 4 | Spark Optimization & Deep Spark | Q7–Q8 | scheduling internals (FIFO/FAIR, locality) · partitioner & co-partitioned sort |
| 5 | ETL — Failures, Late Data, Schema, Fault Tolerance | Q9–Q11 | Delta Sharing distribution · lineaged DQ gates in CI/CD · bad-deploy data repair |
| 6 | Data Modeling | Q12–Q13 | event sourcing vs state tables · 4-4-5 fiscal calendar |
| 7 | Snowflake | Q14–Q15 | Budgets & cost governance · Snowflake Scripting (owner vs caller rights) |
| 8 | Monitoring, Logging & Orchestration | Q16–Q18 | dead-man's-switch meta-monitoring · incident command & postmortem · capstone synthesis incident |

### How to practice
1. Attempt each question on a scratch cell **before** opening the solution. Time yourself: [FAANG] ≈ 30–40 min, [HARD] ≈ 25 min.
2. Say your reasoning OUT LOUD — interviewers grade the narrative, not just the code.
3. Executable solutions self-check with `Q# OK` prints. Talk-track questions end with a whiteboard-style answer you should be able to reproduce.

*Environment note: solutions run on serverless compute. Delta/streaming demos write to fresh scratch dirs under `/Volumes/workspace/default/etl_test_volume/` (`d26_*_nb`) — safe to delete. Streaming needs an explicit `checkpointLocation` (serverless rejects implicit ones); session-window aggregations emit in `append` mode only.*

## Q1 — SQL Advanced: Ranking Rates Without the Small-Sample Trap (Wilson Lower Bound) [FAANG]

**Interviewer (Netflix):** Our A/B dashboard ranks homepage trailer variants by raw click-through rate. The intern demo won last week — it got 1 click on 6 impressions (16.7%) and is beating the incumbent that has 50 clicks on 1,000 impressions (5.0%). Leadership is about to ship it everywhere.

**Requirements**

1. From `q1_variants(variant, impressions, clicks)` compute each variant's CTR and its **Wilson score lower bound** at z = 1.96 in one SQL pass — no UDFs.
2. Show the naive ranking vs the Wilson ranking, and quantify how far the one-click wonder falls.
3. Explain why the bound shrinks for small n: the margin term under the square root and the `1 + z²/n` denominator — and why ranking by the LOWER bound is the standard fix for one-click-wonder bias.
4. Follow-up: why is "filter variants with n < 100" a worse fix than the Wilson bound?

**Sample data** — five variants: n=1000@5%, n=6@16.7%, n=12000@2.5%, n=20@10%, n=1000@4.5%.

In [0]:
# Solution Q1 — Wilson lower bound ranking
from pyspark.sql import functions as F

variants = spark.createDataFrame([
    ("incumbent_5pct", 1000, 50), ("intern_1click", 6, 1), ("big_sample", 12000, 300),
    ("tiny_20", 20, 2), ("challenger", 1000, 45),
], "variant string, impressions bigint, clicks bigint").withColumn(
    "ctr", F.col("clicks") / F.col("impressions"))
variants.createOrReplaceTempView("q1_variants")

wilson_sql = spark.sql("""
SELECT variant, impressions, clicks, ctr,
       (ctr + 1.96*1.96/(2*impressions)
        - 1.96*sqrt(ctr*(1-ctr)/impressions + 1.96*1.96/(4*impressions*impressions)))
       / (1 + 1.96*1.96/impressions) AS wilson_lb
FROM q1_variants ORDER BY wilson_lb DESC
""")
naive_rank = [r["variant"] for r in variants.orderBy(F.desc("ctr")).collect()]
wilson_rank = [r["variant"] for r in wilson_sql.collect()]
lb = {r["variant"]: round(r["wilson_lb"], 5) for r in wilson_sql.collect()}
print("naive ranking:", naive_rank)
print("wilson ranking:", wilson_rank)
print("wilson lower bounds:", lb)
assert naive_rank[0] == "intern_1click", naive_rank            # raw CTR crowns the fluke
assert wilson_rank == ["incumbent_5pct", "challenger", "intern_1click", "tiny_20", "big_sample"], wilson_rank
assert abs(lb["incumbent_5pct"] - 0.03813) < 0.0001, lb
print("Q1 OK — naive #1 is intern_1click (16.7% on n=6); Wilson demotes it to #3; the n=1000 incumbent wins")
print("""
TALK TRACK — why the bound works:
1) The lower bound is p-hat minus a MARGIN that scales ~1/sqrt(n): with n=6 the margin is huge
   (16.7% -> 3.0%), with n=1000 it is tiny (5.0% -> 3.81%). Ranking by lb = 'pessimistic p-hat':
   you only win if the EVIDENCE, not the luck, holds up.
2) The 1/(1+z^2/n) denominator shrinks everything slightly for small n — a second penalty for thin data.
3) Filtering n<100 throws away real information (a 20-exposure variant with 10% CTR is promising;
   Wilson keeps it ranked, just honestly). Filters are a hard cutoff; Wilson is a smooth prior.
4) Production pattern: ORDER BY wilson_lb DESC is how Reddit/Etsy-style 'hot' rankings work.""")

## Q2 — SQL Advanced: Benford's-Law First-Digit Fraud Screen [FAANG]

**Interviewer (Amazon, fraud team):** Accounts-payable analysts suspect a "consulting" vendor of fabricating invoices. Real payment amounts follow **Benford's law** on the first significant digit — P(d) = log₁₀(1 + 1/d) (digit 1 ≈ 30.1%, digit 9 ≈ 4.6%). Fabricators pick amounts that avoid small leading digits.

**Requirements**

1. Screen each vendor group with a **chi-square goodness-of-fit test** on first-digit counts (df = 8, critical value 15.51 at p = 0.05), plus the MAD (mean absolute deviation) of observed vs expected digit shares.
2. Derive expected counts from Benford's law for the group's own n — one pass per group.
3. State a verdict per group and the caveats: when Benford does NOT apply (small n < ~30, uniform price lists, fixed-amount invoices, assigned amounts, capped/rounded data).
4. Follow-up: why is Benford scale-invariant, and why does multiplying every amount by a constant preserve the test?

**Sample data** — 45 honest payments drawn log-uniformly (Benford by construction) vs 20 `acme_consulting` payments clustered in the 700–999 range (digits 7–9 only).

In [0]:
# Solution Q2 — Benford first-digit screen (chi-square + MAD per vendor group)
import math, random

random.seed(42)
honest = [10 ** random.uniform(0, 3) for _ in range(45)]   # log-uniform -> Benford by construction
acme = [random.uniform(700, 999) for _ in range(20)]        # fabricated: leading digits 7-9 only

def first_digit(x):
    return int(str(x).lstrip("0.")[0])

def benford_screen(name, amounts):
    """Chi-square goodness-of-fit on first-digit counts vs Benford expectation.
    df = 9 - 1 = 8 -> critical value 15.51 at p = 0.05."""
    n = len(amounts)
    obs = {d: 0 for d in range(1, 10)}
    for a in amounts:
        obs[first_digit(round(a, 2))] += 1
    exp = {d: n * math.log10(1 + 1 / d) for d in range(1, 10)}
    chi2 = sum((obs[d] - exp[d]) ** 2 / exp[d] for d in range(1, 10))
    mad = sum(abs(obs[d] / n - math.log10(1 + 1 / d)) for d in range(1, 10)) / 9
    verdict = "FLAG for review" if chi2 > 15.51 else "pass"
    print(f"{name}: n={n}  chi2={chi2:.2f}  MAD={mad:.4f}  -> {verdict}")
    return chi2, mad

c_h, m_h = benford_screen("honest_vendors", honest)
c_a, m_a = benford_screen("acme_consulting", acme)
assert c_h < 15.51 and c_a > 15.51, (c_h, c_a)   # expected ~9.9 vs ~112 with this seed
assert m_a > m_h
print(f"Q2 OK — honest chi2 {c_h:.2f} < 15.51 critical; acme_consulting chi2 {c_a:.2f} >> critical -> FLAGGED")
print("""
TALK TRACK — the interview traps:
1) Benford applies to data spanning ORDERS OF MAGNITUDE that result from multiplicative processes.
   It fails on: small n, assigned amounts (invoice numbers), capped fields, uniform price lists.
2) SCALE INVARIANCE: Benford is the only scale-invariant digit law — multiplying every amount by
   7.3 leaves first-digit shares unchanged (log10(1+1/d) is invariant under x -> c*x).
3) It is a SCREEN, not proof: chi2 flags 'unusual', a human investigates the flagged vendor.
4) In SQL: first_digit = CAST(SUBSTRING(CAST(amount AS STRING), 1, 1) AS INT); expected share =
   LOG10(1 + 1/d); group by vendor and compare with a chi2 computed inline.""")

## Q3 — Python: Bounded-Buffer Producer-Consumer — `threading.Condition`, Poison Pills, and the GIL [FAANG]

**Interviewer (Uber):** An ingestion worker polls 4 upstream SaaS APIs (producers) and 2 enrichment workers (consumers) share an in-memory hand-off queue capped at 8. Build it with **raw `threading.Condition`** — not `queue.Queue` — with graceful shutdown via poison pills, and prove exactly-once hand-off. Then explain what the GIL does and does **NOT** protect: why `counter += 1` on a shared list is not atomic, show the lost-update mechanism, and fix it with the lock.

**Requirements**

1. Bounded buffer (cap 8): producers block when full, consumers block when empty — `wait()`/`notify_all()` discipline (why `notify_all` with multiple predicate types).
2. Graceful shutdown: after producers finish, one poison pill **per consumer**; consumers drain remaining real items first.
3. Prove: 200 items handed off exactly once, no duplicates, buffer empty at exit, fail-fast join timeouts (detect deadlock).
4. GIL: demonstrate the lost-update interleaving deterministically; show the live race attempt; show the locked counter is exact.

In [0]:
# Solution Q3 — bounded buffer with Condition + poison pills
import threading, time
from collections import deque

def run_pc(n_items=200, buffer_cap=8, n_producers=4, n_consumers=2):
    buf, lock = deque(), threading.Condition()
    POISON = object()
    consumed, c_counts = [], {}
    counts = [n_items // n_producers] * n_producers
    counts[-1] += n_items - sum(counts)

    def producer(pid, count):
        for i in range(count):
            with lock:
                while len(buf) >= buffer_cap:      # predicate 1: not full
                    lock.wait()
                buf.append((pid, i))
                lock.notify_all()

    def consumer(cid):
        while True:
            with lock:
                while not buf:                     # predicate 2: not empty
                    lock.wait()
                item = buf.popleft()
                lock.notify_all()
            if item is POISON:
                return                            # graceful exit AFTER draining real work
            consumed.append(item)
            c_counts[cid] = c_counts.get(cid, 0) + 1

    t0 = time.time()
    prods = [threading.Thread(target=producer, args=(p, counts[p])) for p in range(n_producers)]
    cons = [threading.Thread(target=consumer, args=(c,)) for c in range(n_consumers)]
    for t in prods + cons:
        t.start()
    for t in prods:
        t.join(timeout=30); assert not t.is_alive(), "producer hung"
    for _ in cons:                                # one pill per consumer
        with lock:
            buf.append(POISON); lock.notify_all()
    for t in cons:
        t.join(timeout=30); assert not t.is_alive(), "consumer hung"
    assert len(buf) == 0, "buffer not drained"
    return consumed, c_counts, time.time() - t0

consumed, c_counts, dur = run_pc()
assert len(consumed) == 200 and len(set(consumed)) == 200   # exactly once, no dups
assert sum(c_counts.values()) == 200
print(f"exactly-once hand-off: {len(consumed)}/200, dup-free, cap=8, split {c_counts}, {dur*1000:.0f}ms")

# --- the GIL does NOT make compound updates atomic ---
import sys
sys.setswitchinterval(0.000001)                 # make preemption aggressive for the demo
race_counter = [0]
def bump_unlocked(per=200_000):
    for _ in range(per):
        race_counter[0] = race_counter[0] + 1    # NOT atomic: LOAD, ADD, STORE_SUBSCR
threads = [threading.Thread(target=bump_unlocked) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()
expected = 8 * 200_000
print(f"unlocked counter: {race_counter[0]:,} vs expected {expected:,} "
      f"(lost {expected - race_counter[0]:,} — races are timing-dependent; if 0 lost, your interpreter got lucky — the mechanism below is deterministic)")

# the lost-update mechanism, deterministically: both threads LOAD the same base
counter = 0
a_read = counter; b_read = counter               # both threads read base 0
a_val = a_read + 1; b_val = b_read + 1            # both compute 1 from the SAME base
counter = a_val; counter = b_val                  # A stores 1, then B's store OVERWRITES it
assert counter == 1, "two increments advanced the counter by exactly 1 — the lost update"

safe_counter, safe_lock = [0], threading.Lock()
def bump_locked(per=100_000):
    for _ in range(per):
        with safe_lock:                           # the compound read-modify-write becomes atomic
            safe_counter[0] += 1
threads = [threading.Thread(target=bump_locked) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()
assert safe_counter[0] == 800_000, safe_counter[0]
print(f"locked counter: exactly {safe_counter[0]:,}")
print("Q3 OK — Condition buffer exactly-once; unlocked increment loses updates (shown deterministically); lock fixes it")
print("""
TALK TRACK — the interview traps:
1) WHY notify_all: producers and consumers wait on the SAME condition object with DIFFERENT
   predicates (full vs empty). A bare notify() can wake the wrong predicate class -> lost wakeup
   -> deadlock. notify_all is the safe default; separate Condition objects per side are the optimization.
2) POISON PILL: consumers must check AFTER acquiring a real item, so pills are appended AFTER
   producers finish; one pill per consumer, then notify_all.
3) GIL: it guarantees bytecode-level atomicity of SINGLE ops, not of read-modify-write sequences.
   It also makes CPU-bound threads slower (context switching overhead), while IO/lock waits release it.
4) join(timeout) + is_alive() is your deadlock detector in interviews and in prod.""")

## Q4 — Python: Floating-Point Error in Distributed Aggregation — Kahan, Tree Reduction, Decimal [FAANG]

**Interviewer (Meta):** Your Spark job sums a million rows of $0.10-priced items and finance's total is off by a hair — and the SAME query returns slightly different cents run to run. Explain and fix.

**Requirements**

1. Reproduce: naive left-to-right summation of 1,000,000 × 0.1 vs the exact 100,000.0 — quantify the error and explain WHY (binary fractions, rounding at every add).
2. Fix ladder: **Kahan compensated summation** (track the lost low-order bits in `c`), **tree reduction** (why partial sums reduce error: depth log n vs n), `math.fsum`, and `Decimal` (exact decimal arithmetic — the money answer).
3. Show catastrophic cancellation: `(1e16 + 1) - 1e16` in float vs Decimal.
4. Explain the DISTRIBUTED angle: why partial sums computed in different partition orders give different totals (your nondeterministic cents), and the production fix (round-to-cents at the money boundary, DECIMAL types, or exact accumulators).

**Sample data** — 1M items at 0.1, plus a cancellation pair.

In [0]:
# Solution Q4 — floating point error in aggregation
import math
from decimal import Decimal

def naive_sum(xs):
    s = 0.0
    for x in xs:
        s += x
    return s

def kahan_sum(xs):
    s, c = 0.0, 0.0
    for x in xs:
        y = x - c                # apply the correction from the previous iteration
        t = s + y                # the (rounded) new total
        c = (t - s) - y          # the low-order bits lost in t — recover them
        s = t
    return s

def tree_sum(xs, parts=64):
    n, step = len(xs), len(xs) // parts
    partials = [naive_sum(xs[i * step:(i + 1) * step]) for i in range(parts)]
    return naive_sum(partials)   # error grows with SUM DEPTH, not item count

xs = [0.1] * 1_000_000
target = 100_000.0
naive = naive_sum(xs)
kahan = kahan_sum(xs)
tree = tree_sum(xs)
fsum = math.fsum(xs)
dec_val = Decimal("0.1") * 1_000_000   # exact: 0.1 is exactly representable in base 10
print(f"naive err={abs(naive-target):.3e}  kahan err={abs(kahan-target):.3e}  "
      f"tree err={abs(tree-target):.3e}  fsum err={abs(fsum-target):.3e}  decimal err={abs(float(dec_val)-target):.3e}")
assert abs(naive - target) > 1e-9        # naive IS wrong — order-dependent error
assert abs(kahan - target) < 1e-9        # Kahan: near-exact
assert abs(tree - target) < abs(naive - target)  # depth log(n): far better than naive
assert abs(fsum - target) < 1e-9         # fsum: exact (Shewchuk algorithm)
assert float(dec_val) == target          # Decimal: exact by construction

big = 1e16
assert (big + 1.0) - big == 0.0          # 1.0 is BELOW the ulp of 1e16 -> absorbed
assert float(Decimal("1e16") + Decimal(1) - Decimal("1e16")) == 1.0   # Decimal keeps it
print("cancellation: float (1e16 + 1) - 1e16 == 0.0  |  Decimal == 1.0")

vals = [0.1, 1e12, -1e12, 0.1, 0.1, 1e13, -1e13] * 5
fwd, rev = naive_sum(vals), naive_sum(vals[::-1])
print(f"same data, different order: forward={fwd!r} reverse={rev!r} equal={fwd == rev}")
assert fwd != rev                         # why your distributed total isn't reproducible
print("Q4 OK — naive drifts (order-dependent); Kahan/tree/fsum/Decimal exact; 1e16+1-1e16 loses 1.0 in float")
print("""
TALK TRACK — the interview traps:
1) 0.1 is 1/10 -> INFINITE binary fraction -> stored rounded. Every add rounds again: errors
   accumulate ~ proportional to n * ulp (left fold) vs log(n) * ulp (tree).
2) Kahan keeps a correction term c: (t - s) - y recovers the bits the last add rounded away.
3) SPARK angle: SUM over partitions = tree reduction, but the ORDER of partial sums varies with
   partitioning/scheduling -> different low-order bits -> different cents. It is not a bug — it is IEEE 754.
4) MONEY FIX: DECIMAL(p,2) columns, round once at the reporting boundary, or exact accumulation
   (fsum-style) — never round intermediate partials.""")

## Q5 — PySpark Coding: Event-Time Sessionization with the Built-In `session_window` [FAANG]

**Interviewer (DoorDash):** Day 19 you built arbitrary-gap sessionization with `flatMapGroupsWithState`. Spark 3.2+ ships a native **`session_window`**. Rebuild the clickstream sessionization with the built-in on a *streaming* DataFrame.

**Requirements**

1. Group events by user into sessions that close after a **10-minute** inactivity gap, watermark 30 minutes, on a streaming source (Parquet + `trigger(availableNow=True)`).
2. Session rows must emit only when the **watermark passes the session end** — and explain why session aggregations REJECT `update` output mode (state cannot be retracted) and emit in `append` only.
3. Use a late "clock" event (u3 @ 11:10) to push the watermark to 10:40 so the early sessions close — and show u3's own session is still HELD OPEN (session end 11:20 > watermark).
4. Follow-up: when do you still need `flatMapGroupsWithState` (dynamic gaps, per-session custom state/outputs, emitting mid-session)?

**Sample events** — u1 10:00, 10:05, 10:20 (gap > 10 min → new session); u2 10:02, 10:09; u3 11:10 (watermark driver).

In [0]:
# Solution Q5 — session_window on a streaming DataFrame (validated exact notebook flow)
from pyspark.sql import functions as F
import datetime as dt

BASE = "/Volumes/workspace/default/etl_test_volume"
try:  # idempotent re-runs: clean checkpoint + memory sink table
    dbutils.fs.rm(f"{BASE}/d26_q5_events_nb", True)
    dbutils.fs.rm(f"{BASE}/d26_q5_ckpt_nb", True)
except Exception:
    pass
try:
    spark.sql("DROP TABLE IF EXISTS q5_nb_sessions")
except Exception:
    pass

ev = spark.createDataFrame([
    ("u1", dt.datetime(2026, 11, 18, 10, 0)),
    ("u1", dt.datetime(2026, 11, 18, 10, 5)),
    ("u1", dt.datetime(2026, 11, 18, 10, 20)),   # gap > 10 min -> NEW session
    ("u2", dt.datetime(2026, 11, 18, 10, 2)),
    ("u2", dt.datetime(2026, 11, 18, 10, 9)),   # same session
    ("u3", dt.datetime(2026, 11, 18, 11, 10)),  # advances watermark to 10:40
], "user string, ts timestamp")
ev.write.mode("overwrite").parquet(f"{BASE}/d26_q5_events_nb")

stream = (spark.readStream
          .schema("user string, ts timestamp")
          .parquet(f"{BASE}/d26_q5_events_nb")
          .withWatermark("ts", "30 minutes")
          .groupBy(F.session_window("ts", "10 minutes").alias("sess"), "user")
          .count())

q = (stream.writeStream
     .format("memory").queryName("q5_nb_sessions")
     .outputMode("append")                                   # sessions reject update mode
     .option("checkpointLocation", f"{BASE}/d26_q5_ckpt_nb")  # serverless: no implicit checkpoints
     .trigger(availableNow=True).start())
q.awaitTermination()

res = spark.table("q5_nb_sessions").select("user", "sess.start", "sess.end", "count").collect()
for r in sorted(res, key=lambda r: (r["user"], r["start"])):
    print(f"  {r['user']}  [{r['start']:%H:%M} -> {r['end']:%H:%M}]  n={r['count']}")
by_user = {}
for r in res:
    by_user.setdefault(r["user"], []).append(r["count"])
assert sorted(by_user.get("u1", [])) == [1, 2], by_user
assert by_user.get("u2", []) == [2], by_user
assert "u3" not in by_user, "u3 must still be held open (session end 11:20 > watermark 10:40)"
print("Q5 OK — u1: 2 sessions (2+1 events); u2: 1 session (2); u3 correctly held open")
print("""
TALK TRACK — the interview traps:
1) Sessions are MUTABLE state: an event can MERGE two sessions into one. Since state cannot be
   retracted downstream, only append mode is allowed (update throws STREAMING_OUTPUT_MODE error)
   and a row only EMITS when the watermark passes the session end — that is why u3 is invisible.
2) The watermark driver trick: a later high-timestamp event (u3 @ 11:10) advances the watermark to
   10:40, closing the 10:00-10:30 sessions. In production this happens naturally as new data flows.
3) Gap-based merging is built in (10 minutes); for DYNAMIC gaps (e.g. gap depends on the user),
   mid-session emissions, or per-session custom payloads you still need flatMapGroupsWithState.
4) availableNow + explicit checkpointLocation is the batch-style streaming pattern that runs on
   serverless; classic implicit checkpoints are rejected there.""")

## Q6 — PySpark Coding: Proving Incremental Rollup Maintenance with a Correctness Harness [HARD]

**Interviewer (Stripe):** Day 22 covered *when* materialized views can incrementally refresh. Now build it yourself: maintain a daily revenue rollup (SUM, COUNT) incrementally via the **Delta Change Data Feed**, and PROVE it equals a full recompute after each append — an equality harness is the interview differentiator. Then show the trap: **median is NOT delta-maintainable** — prove the rollup's median column silently goes stale.

**Requirements**

1. Create the source with **CDF enabled at creation** (why `startingVersion=0` raises `DELTA_MISSING_CHANGE_DATA` if you enable it after writing).
2. Capture the source version right after the append (`DESCRIBE HISTORY`), read the CDF from that version, and MERGE only the delta into the rollup (SUM += delta, COUNT += 1).
3. Equality harness: incremental result == full recompute — assert it, every append.
4. The trap: algebraic measures (SUM/COUNT via a merge function) are incrementally maintainable; **holistic** measures (median/percentile/distinct-count) are not — show the rollup median (75.0) vs true median (50.0) after the append, and state the rebuild path.

**Sample data** — orders across 3 days, then one late append on 2026-11-10.

In [0]:
# Solution Q6 — incremental rollup via CDF + equality harness (validated exact notebook flow)
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/etl_test_volume"
try:  # idempotent re-runs
    dbutils.fs.rm(f"{BASE}/d26_q6_src_nb", True)
    dbutils.fs.rm(f"{BASE}/d26_q6_rollup_nb", True)
except Exception:
    pass
SRC = f"{BASE}/d26_q6_src_nb"; ROLL = f"{BASE}/d26_q6_rollup_nb"

# 1) CDF enabled AT CREATION (delta.enableChangeDataFeed) — not retroactive
d0 = spark.createDataFrame([
    ("2026-11-10", 100), ("2026-11-10", 50), ("2026-11-11", 200), ("2026-11-12", 75)],
    "order_date string, amount double")
d0.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").save(SRC)

full0 = (spark.read.format("delta").load(SRC)
         .groupBy("order_date")
         .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders"),
              F.expr("percentile_cont(0.5) WITHIN GROUP (ORDER BY amount)").alias("med")))
full0.write.format("delta").mode("overwrite").save(ROLL)

# 2) new business arrives; capture the version of THIS append
spark.createDataFrame([("2026-11-10", 10.0)], "order_date string, amount double") \
     .write.format("delta").mode("append").save(SRC)
v_append = spark.sql(f"DESCRIBE HISTORY delta.`{SRC}`").select("version").agg(F.max("version")).collect()[0][0]

# 3) read only the delta (inserts) from the append version
cdc = (spark.read.format("delta")
       .option("readChangeFeed", "true").option("startingVersion", v_append)
       .load(SRC)
       .where(F.col("_change_type") == "insert"))
assert cdc.count() == 1
cdc.createOrReplaceTempView("q6_nb_delta")
spark.sql(f"""
MERGE INTO delta.`{ROLL}` r USING q6_nb_delta s ON r.order_date = s.order_date
WHEN MATCHED THEN UPDATE SET r.revenue = r.revenue + s.amount, r.orders = r.orders + 1
WHEN NOT MATCHED THEN INSERT (order_date, revenue, orders) VALUES (s.order_date, s.amount, 1)
""")

# 4) EQUALITY HARNESS: incremental == full recompute (run after EVERY append)
inc = {r["order_date"]: (r["revenue"], r["orders"]) for r in spark.read.format("delta").load(ROLL).collect()}
full = {r["order_date"]: (r["revenue"], r["orders"])
        for r in (spark.read.format("delta").load(SRC).groupBy("order_date")
                  .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")).collect())}
print("  incremental:", inc)
print("  recompute:   ", full)
assert inc == full, "equality harness FAILED — incremental diverged from truth"
print("  equality harness OK — incremental MERGE == full recompute (algebraic: SUM/COUNT)")

# 5) THE TRAP: median is HOLISTIC — no merge function exists; the column went stale
med_rollup = {r["order_date"]: r["med"] for r in spark.read.format("delta").load(ROLL).select("order_date", "med").collect()}
med_true = {r["order_date"]: r["med"] for r in (spark.read.format("delta").load(SRC).groupBy("order_date")
            .agg(F.expr("percentile_cont(0.5) WITHIN GROUP (ORDER BY amount)").alias("med")).collect())}
print("  stale rollup median:", med_rollup["2026-11-10"], "| true median:", med_true["2026-11-10"])
assert med_rollup["2026-11-10"] != med_true["2026-11-10"], "median should have drifted after the append"
print("Q6 OK — SUM/COUNT maintained incrementally & PROVEN equal; median is not delta-maintainable -> rebuild required")
print("""
TALK TRACK — the interview traps:
1) CDF is per-table and enabled AT CREATION; enabling later means early versions have no change
   data -> startingVersion=0 raises DELTA_MISSING_CHANGE_DATA.
2) Version-relative reads: capture the post-append version, read CDF FROM that version — never
   re-read the whole feed. _change_type in (insert, update_postimage, delete) maps onto the MERGE.
3) The equality harness is the trust builder: after every incremental merge, diff against a full
   recompute on a sample window. This is exactly what Databricks MVs do internally for
   incrementally-refreshable plans (Day 22).
4) HOLISTIC measures (median, percentile, count-distinct) cannot be merged from deltas alone —
   keep a rebuild path (or a sketch like Day 24's HLL for approximations).""")

## Q7 — Spark Optimization: Scheduling Internals — FIFO vs FAIR Pools, Locality & Delay Scheduling [HARD]

**Interviewer (Databricks platform team):** Two teams share one cluster: the nightly ETL hogs every core, and BI's 5-minute queries starve until morning. Derive the scheduler.

**Requirements**

1. **FIFO vs FAIR**: simulate FIFO (head-of-queue job takes every free slot — show the starvation) vs FAIR pools with `minShare` and `weight` (weight 3:1 across 10 slots → 7/3 after both minShares are met).
2. Where pool config lives: `spark.scheduler.mode=FAIR`, `spark.scheduler.allocationFile`, `fairscheduler.xml` (`<pool name=... minShare=... weight=...>`), and what's locked on serverless.
3. **Locality levels** — PROCESS_LOCAL → NODE_LOCAL → NO_PREF → RACK_LOCAL → ANY — and **delay scheduling**: a task prefers NODE_LOCAL; if the wait exceeds `spark.locality.wait` (3s) it relaxes. Show the 3s-ANY vs 5s-NODE_LOCAL trade.
4. Why speculative execution respects locality, and how pools isolate noisy neighbors (vs separate clusters).

In [0]:
# Solution Q7 — scheduling internals, simulated deterministically
def fifo_share(pending, slots):
    """FIFO: head of the queue takes EVERY free slot; later jobs starve."""
    alloc, free = {}, slots
    for job, ntasks in pending:
        take = min(ntasks, free)
        alloc[job] = take
        free -= take
    return alloc

fifo = fifo_share([("ad_hoc_big_scan", 10), ("analyst_5min_query", 4)], 10)
assert fifo == {"ad_hoc_big_scan": 10, "analyst_5min_query": 0}
print("FIFO (big job first):", fifo, "-> the analyst query STARVES")

def fair_share(pools, slots):
    """FAIR: (1) meet minShare first, (2) then one slot at a time to the pool with the
    LOWEST running/weight ratio (weighted fair queuing)."""
    running = {p: 0 for p in pools}
    for _ in range(slots):
        under = [p for p in pools if running[p] < pools[p]["min"]]
        if under:
            running[under[0]] += 1
            continue
        p = min(pools, key=lambda p: running[p] / pools[p]["weight"])
        running[p] += 1
    return running

pools = {"etl_pool": {"weight": 3, "min": 2}, "bi_pool": {"weight": 1, "min": 2}}
fair = fair_share(pools, 10)
print("FAIR pools:", fair)
assert fair == {"etl_pool": 7, "bi_pool": 3}, fair     # minShare 2+2 met, then 3:1 on the rest

def delay_scheduling(locality_wait=3, node_local_free_at=5):
    """Task prefers NODE_LOCAL (busy until t=5). It waits while its unfulfilled
    preference is younger than locality_wait, then relaxes to ANY."""
    t = 0
    while True:
        if t >= node_local_free_at:
            return "NODE_LOCAL", t
        if t >= locality_wait:
            return "ANY", t
        t += 1

lvl, t = delay_scheduling(locality_wait=3)
lvl2, t2 = delay_scheduling(locality_wait=10)
print(f"delay scheduling: wait=3s -> launched {lvl} at t={t}s (relaxed); wait=10s -> {lvl2} at t={t2}s (locality kept)")
assert (lvl, t) == ("ANY", 3) and (lvl2, t2) == ("NODE_LOCAL", 5)

try:
    print("spark.scheduler.mode =", spark.conf.get("spark.scheduler.mode"))
except Exception as e:
    print(f"spark.scheduler.mode: NOT AVAILABLE on serverless ({type(e).__name__}) — pool config lives in cluster conf / fairscheduler.xml")
print("Q7 OK — FIFO starves late jobs; FAIR honors minShare then weight 3:1 -> 7/3; locality relaxes after the wait")
print("""
TALK TRACK — the interview traps:
1) FAIR allocation = minShare FIRST (guaranteed floor), then weighted fair queuing on the
   remainder (next slot to the lowest running/weight). minShare is anti-starvation; weight is
   proportional entitlement. 8/2 (all remainder to the heavy pool) is the WRONG answer — the
   weighted-ratio rule is what equalizes entitlement.
2) Locality: PROCESS_LOCAL (same executor) > NODE_LOCAL (same host, cached/HDFS block) >
   NO_PREF > RACK_LOCAL > ANY. Delay scheduling trades launch latency for data locality —
   default wait 3s, per-level, and it RESETS when a task launches locally.
3) Speculative execution: the speculator launches a copy of a straggler ONLY when a free slot
   appears on a worker that ALREADY has locality for that task — locality-preserving.
4) Serverless locks fleet-level scheduler confs; on classic clusters set mode+allocationFile
   (or pool per team via spark.pool tag in jobs). Pools share one cluster's fixed capacity —
   no isolation on memory/errors though; separate clusters isolate those.""")

## Q8 — Spark Optimization: Partitioners & Co-Partitioned Sort (the Secondary-Sort Pattern) [HARD]

**Interviewer (Apple):** Why does `repartition(4, "key")` colocate every row of a key into one partition while `repartition(4)` does not? And why is `repartition(key).sortWithinPartitions(key, ts)` the classic MapReduce "secondary sort" brought forward to Spark?

**Requirements**

1. Derive **hash vs range partitioners**: `repartition(n, col)` = HashPartitioner on MurmurHash(key) mod n (all rows of a key → one partition); `repartitionByRange` = sampled range splits (contiguous key ranges, ordered).
2. **Co-partitioned joins**: both sides repartitioned on the join key with the SAME width → same key lands together on both sides → sort-merge join without a second shuffle. Prove with `spark_partition_id()`.
3. **Secondary sort**: per-partition sorted by (key, ts) — verify partitions come out sorted without a global sort; explain why sortWithinPartitions beats repartition+sort (no global barrier) and where AQE coalescing interacts.
4. Follow-up: why is one partition per key BAD for skewed keys (single hot partition), and what you do instead (salting/AQE skew join).

In [0]:
# Solution Q8 — partitioner behavior, verified with spark_partition_id
from pyspark.sql import functions as F

rows = [(k, t, v) for k in ["a", "b", "c", "d", "e"] for t in [3, 1, 2] for v in [10 * t]]
df = spark.createDataFrame(rows, "key string, ts int, val int")

# 1) HASH partitioner on key: every key -> exactly ONE partition id
recs = (df.repartition(4, "key").sortWithinPartitions("key", "ts")
        .withColumn("pid", F.spark_partition_id()).collect())
pid_by_key = {}
for r in recs:
    pid_by_key.setdefault(r["key"], set()).add(r["pid"])
assert all(len(v) == 1 for v in pid_by_key.values()), pid_by_key
print("hash colocates: each key lands in exactly 1 partition:", {k: list(v)[0] for k, v in pid_by_key.items()})

# 2) SECONDARY SORT: every partition is sorted by (key, ts) — no global sort anywhere
for pid in {r["pid"] for r in recs}:
    part = [(r["key"], r["ts"]) for r in recs if r["pid"] == pid]
    assert part == sorted(part), f"partition {pid} not sorted"
print("secondary sort: every partition verified sorted by (key, ts)")

# 3) CO-PARTITIONED JOIN: both sides hashed on key, same width -> same pid per key
pl = {r["key"]: r["pidL"] for r in df.repartition(4, "key").withColumn("pidL", F.spark_partition_id()).collect()}
pr = {r["key"]: r["pidR"] for r in df.repartition(4, "key").withColumn("pidR", F.spark_partition_id()).collect()}
assert pl == pr, "join sides must colocate the same key -> same partition"
print("co-partitioned join: same key -> same pid on BOTH sides (sort-merge without re-shuffle)")

# 4) RANGE partitioner: contiguous, ordered key ranges (uses sampling, not hashing)
rr = (df.select("key").distinct().repartitionByRange(3, "key")
      .withColumn("pid", F.spark_partition_id()).orderBy("key").collect())
range_layout = {}
for r in rr:
    range_layout.setdefault(r["pid"], []).append(r["key"])
print("range splits (ordered, contiguous):", range_layout)
allk = [k for pid in sorted(range_layout) for k in range_layout[pid]]
assert allk == sorted(allk), "range partitions must hold contiguous key ranges"
print("Q8 OK — hash colocates keys (1 pid per key, both join sides agree); range gives sorted contiguous splits; per-partition (key,ts) sort verified")
print("""
TALK TRACK — the interview traps:
1) repartition(4) with NO column is ROUND-ROBIN — arbitrary spread, no key guarantee.
   repartition(4, 'key') is MurmurHash(key) % 4: deterministic, same key -> same partition.
2) Secondary sort = sort WITHIN partitions after key-hash repartition: streaming-friendly
   (no global barrier), exactly the MapReduce 'sort by key then value' pattern. Per-partition
   sort is bounded by partition size, not table size.
3) Co-partitioned joins: if both sides arrive hash-partitioned on the join key with the same
   number of partitions, Spark's SMJ skips the exchange. Different widths or encodings ->
   one side re-shuffles (look for Exchange in the plan).
4) SKEW: all rows of a hot key in ONE partition is the single-hot-partition problem —
   AQE skew join splits it, or you salt the key. Colocation is a feature AND a hazard.""")

## Q9 — ETL: Delta Sharing — Distributing a Gold Dataset to External Consumers [FAANG]

**Interviewer (Adidas, data marketplace):** A partner wants your gold product catalog daily and incrementally — **without** giving them credentials into your cloud account, and without exporting CSV drops. Design the end-to-end Delta Sharing flow.

**Requirements**

1. **The protocol**: why Delta Sharing is an open REST protocol (Parquet files + Delta transaction log semantics delivered via a REST API), and what that implies for consumer freedom (any client, any cloud, even Snowflake/Trino readers).
2. **Provider side**: `CREATE SHARE`, `ALTER SHARE ... ADD TABLE`, `CREATE RECIPIENT` (+ token activation policy), grants — write the actual DDL.
3. **Incremental refresh**: how a consumer streams changes (CDF query history / change feed through the share) instead of re-reading the full table; what the recipient CAN and CANNOT do (SELECT only — no time travel beyond shared history, no writes, no re-share).
4. **Governance & cost**: token rotation/revocation, who pays (provider pays storage + serving; consumer pays THEIR compute), audit via system tables; vs Snowflake secure shares (Day 19) and when to prefer a Marketplace listing.

Talk-track + DDL/SQL examples (no live recipient exists in this workspace).

In [0]:
# Solution Q9 — Delta Sharing: provider DDL + consumer flow (talk track; not executed against a live recipient)
provider_ddl = """
-- PROVIDER side (Databricks workspace with Unity Catalog)
CREATE SHARE IF NOT EXISTS retail_partner_share COMMENT 'Gold catalog for partner X';

ALTER SHARE retail_partner_share
  ADD TABLE main.gold.products       -- recipient gets SELECT-level reads only
  ADD TABLE main.gold.daily_sales;

CREATE RECIPIENT IF NOT EXISTS partner_x
  USING ID 'partner-x-external-id'   -- or authentication_type = TOKEN for open sharing
  COMMENT 'Partner X analytics team';

-- activation: recipient gets a one-time activation link/token; Databricks-to-Databricks
-- recipients authenticate with their workspace identity automatically.
GRANT SELECT ON SHARE retail_partner_share TO RECIPIENT partner_x;

-- rotate/expire access instantly (no data copy involved)
ALTER RECIPIENT partner_x SET TOKEN_LIFETIME = '30 DAYS';
"""

consumer_code = """
# CONSUMER side — any client that speaks the open REST protocol
from databricks.sdk import WorkspaceClient
w = WorkspaceClient(profile="partner_share")

# full snapshot through the share (no cloud credentials to the provider account)
df = (w.statement_execution
     .execute_statement("SELECT * FROM partner_x.retail_partner_share.gold.daily_sales")
     .result)

# INCREMENTAL: query the change feed through the share — pull only what changed
changes = ("SELECT * FROM table_changes("
           "'partner_x.retail_partner_share.gold.daily_sales', 12)  "  # version 12 onwards
           "WHERE _change_type IN ('insert','update_postimage')")
"""

print(provider_ddl)
print(consumer_code)
assert "CREATE SHARE" in provider_ddl and "ADD TABLE" in provider_ddl
assert "GRANT SELECT ON SHARE" in provider_ddl and "table_changes" in consumer_code
print("Q9 OK (talk track) — share->recipient->grant flow, CDF-through-share incremental pulls, token rotation")
print("""
TALK TRACK — the interview traps:
1) OPEN PROTOCOL: the share is served over a REST API that delivers Parquet + Delta log
   semantics; consumers are NOT locked to Databricks (Spark OSS connector, Trino, Snowflake
   via the Delta Sharing connector, pandas). Data does NOT leave your lakehouse as copies —
   the provider serves pre-signed short-lived URLs to files on read.
2) RECIPIENT RIGHTS: SELECT through the share only. No writes, no re-sharing, no time travel
   before the share existed. Access revocation is instant (token invalidation) because files
   are addressed via short-lived signed URLs, not copied buckets.
3) COST MODEL: provider pays storage + egress on reads; consumer pays their own compute.
   A Marketplace listing adds discovery/billing on top of the same machinery.
4) VS SNOWFLAKE SECURE SHARES (Day 19): Snowflake sharing requires the CONSUMER to be on
   Snowflake; Delta Sharing is engine-agnostic. Both avoid data duplication — the deciding
   question is 'what engines must the partner run?'.""")

## Q10 — ETL: Lineage-Aware Data-Quality Gates in CI/CD [HARD]

**Interviewer (Spotify):** A bronze quality failure shipped to production last month because the gold build didn't know its *grandparent's* gate had failed — each layer only checked its own input. Build **lineage-propagated gating**.

**Requirements**

1. A **gates ledger** (append-only Delta: run_id, table, rule, passed, fail_rows) and a **quarantine sink** for bad rows — the two durable artifacts of every gate evaluation.
2. A **lineage walk** (child → parents, transitively) that blocks a downstream build if ANY ancestor's latest gate for the run failed. `gold` depends on `silver` depends on `bronze`.
3. Demo run 1 (clean data): all gates pass, gold builds. Run 2 (a negative amount in bronze): bronze gate FAILS, the bad row is quarantined, the gold build is **blocked by the lineage walk**, and gold is untouched.
4. Where does this live? Deploy-time gate in the CI/CD pipeline (Declarative Automation Bundles `databricks bundle deploy` calling the gate) vs runtime gate in the job — argue both.

**Sample data** — run 1: o1/o2 valid; run 2: o3 = −10.0 invalid, o4 valid.

In [0]:
# Solution Q10 — gates ledger + lineage walk (validated exact notebook flow)
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/etl_test_volume"
try:  # idempotent re-runs: gates/quarantine are APPEND-only sinks -> start clean
    for d in ["d26_q10_bronze_nb", "d26_q10_silver_nb", "d26_q10_gold_nb",
              "d26_q10_gates_nb", "d26_q10_quarantine_nb"]:
        dbutils.fs.rm(f"{BASE}/{d}", True)
except Exception:
    pass
BR = f"{BASE}/d26_q10_bronze_nb"; SV = f"{BASE}/d26_q10_silver_nb"; GD = f"{BASE}/d26_q10_gold_nb"
GATES = f"{BASE}/d26_q10_gates_nb"; QUAR = f"{BASE}/d26_q10_quarantine_nb"
LINEAGE = {"gold": ["silver"], "silver": ["bronze"], "bronze": []}   # child -> parents

def gate(run_id, table, rows, rule_fn, rule_name):
    """Evaluate one DQ rule: quarantine failures, record the gate, return (passed, good)."""
    bad = rows.filter(~rule_fn)
    good = rows.filter(rule_fn)
    if bad.count() > 0:
        bad.withColumn("run_id", F.lit(run_id)).withColumn("rule", F.lit(rule_name)) \
           .write.format("delta").mode("append").save(QUAR)
    passed = bad.count() == 0
    spark.createDataFrame([(run_id, table, rule_name, passed, int(bad.count()))],
        "run_id string, table string, rule string, passed boolean, fail_rows int") \
         .write.format("delta").mode("append").save(GATES)
    return passed, good

def lineage_blocked(run_id, target):
    """Walk child -> parents transitively; block if any ancestor's LATEST gate for the run failed."""
    to_check, blocked = list(LINEAGE[target]), []
    while to_check:
        t = to_check.pop()
        to_check.extend(LINEAGE[t])
        latest = (spark.read.format("delta").load(GATES)
                  .where((F.col("run_id") == run_id) & (F.col("table") == t))
                  .orderBy(F.col("fail_rows").desc()).limit(1).collect())
        if latest and not latest[0]["passed"]:
            blocked.append(t)
    return blocked

# ---- run 1: clean data end-to-end ----
bronze_rows = spark.createDataFrame([("o1", 100.0), ("o2", 50.0)], "order_id string, amount double")
ok_b, good_b = gate("r1", "bronze", bronze_rows, F.col("amount") > 0, "amount_positive")
good_b.write.format("delta").mode("overwrite").save(BR)
blocked = lineage_blocked("r1", "gold")
assert blocked == [], blocked
silver_rows = spark.read.format("delta").load(BR).withColumn("amount2", F.col("amount") * 1.1)
ok_s, good_s = gate("r1", "silver", silver_rows, F.col("amount2") > 0, "amount2_positive")
assert ok_s
blocked = lineage_blocked("r1", "gold")
assert blocked == [], blocked
good_s.write.format("delta").mode("overwrite").save(SV)
spark.read.format("delta").load(SV).write.format("delta").mode("overwrite").save(GD)
assert spark.read.format("delta").load(GD).count() == 2
print("run 1: all gates pass; gold built with 2 rows")

# ---- run 2: bronze rule violated -> gold MUST be blocked by the lineage walk ----
bad_bronze = spark.createDataFrame([("o3", -10.0), ("o4", 20.0)], "order_id string, amount double")
ok_b2, _ = gate("r2", "bronze", bad_bronze, F.col("amount") > 0, "amount_positive")
assert not ok_b2, "bronze gate must fail on a negative amount"
blocked = lineage_blocked("r2", "gold")
print("run 2: bronze gate FAILED; lineage walk blocks:", blocked)
assert blocked == ["bronze"], blocked
q = spark.read.format("delta").load(QUAR).where(F.col("run_id") == "r2")
assert q.count() == 1 and q.collect()[0]["order_id"] == "o3"   # only the bad row, kept for triage
assert spark.read.format("delta").load(GD).count() == 2          # gold untouched
print("Q10 OK — gate recorded, bad row quarantined, gold build BLOCKED by lineage walk, gold untouched")
print("""
TALK TRACK — the interview traps:
1) The ledger is APPEND-ONLY and keyed by run_id: gates are evidence for audits, not just
   booleans. fail_rows gives trend dashboards ('is bronze getting dirtier?').
2) Per-layer checks miss TRANSITIVE failures — the lineage walk (child -> parents) is the fix.
   In prod, read the lineage from system tables / UC lineage instead of a hardcoded dict.
3) DEPLOY-TIME vs RUNTIME: deploy gates block a bad TRANSFORM from shipping (fast, in CI/CD,
   via databricks bundle deploy); runtime gates block bad DATA from flowing (every run, every
   batch). You need both — Day 25's tripwires are the runtime leg; this is the lineage leg.
4) Quarantine, don't drop: the bad row plus its run_id is what the triage query and the
   re-processing job need.""")

## Q11 — ETL: Repairing a Bad Deploy — Blast-Window Detection & Version-Guarded Replay [FAANG]

**Interviewer (Airbnb):** A deploy inverted a commission expression for exactly one hour (14:00–15:00) before someone rolled it back. `revenue = gross * (1 − commission)` shipped as `* commission` — gold's hour-14 revenue is 20.0 where it should be 180.0. Write the repair runbook.

**Requirements**

1. Find the **blast window** from the run log (a `system.lakeflow.job_run_timeline` stand-in: job, start, end, status) — the BAD_DEPLOY row gives hours 14–14.
2. Confirm the damage, then repair by re-transforming ONLY the blast window and MERGE-ing back — no full rebuild, no stopping consumers.
3. **Prove the repair**: (a) the repaired table equals a full recompute, and (b) via time travel (`versionAsOf` pre-repair) exactly ONE hour's row changed.
4. Follow-ups: why MERGE not overwrite; what downstream consumers need to know (revenue restatement, caches, alerts that already fired); how the same runbook generalizes to a whole-day window or a wrong-join-key deploy.

**Sample data** — 4 hourly events; the bad MERGE touched only hour 14.

In [0]:
# Solution Q11 — blast-window repair (validated exact notebook flow)
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/etl_test_volume"
try:  # idempotent re-runs
    dbutils.fs.rm(f"{BASE}/d26_q11_src_nb", True)
    dbutils.fs.rm(f"{BASE}/d26_q11_gold_nb", True)
    dbutils.fs.rm(f"{BASE}/d26_q11_runlog_nb", True)
except Exception:
    pass
SRC = f"{BASE}/d26_q11_src_nb"; G11 = f"{BASE}/d26_q11_gold_nb"; RUNS = f"{BASE}/d26_q11_runlog_nb"

ev = spark.createDataFrame([
    ("h13", 13, 100.0), ("h14", 14, 200.0), ("h15", 15, 300.0), ("h16", 16, 400.0),
], "hour string, hr int, amount double")
ev.write.format("delta").mode("overwrite").save(SRC)

def transform(commission, hours=None):
    """The transform as CURRENTLY deployed (correct). hours limits the blast window."""
    df = spark.read.format("delta").load(SRC)
    if hours:
        df = df.where(F.col("hr").between(hours[0], hours[1]))
    return (df.groupBy("hour").agg(F.sum("amount").alias("gross"))
              .withColumn("revenue", F.round(F.col("gross") * (1 - commission), 2)))

# v1 good deploy (all hours), then the BAD deploy: a scoped MERGE with the inverted expression
transform(0.10).write.format("delta").mode("overwrite").save(G11)
bad = transform(0.90, hours=(14, 14))                       # deployed as 1-commission vs commission
bad.createOrReplaceTempView("q11_bad")
spark.sql(f"""
MERGE INTO delta.`{G11}` g USING q11_bad b ON g.hour = b.hour
WHEN MATCHED THEN UPDATE SET g.gross = b.gross, g.revenue = b.revenue
WHEN NOT MATCHED THEN INSERT *
""")

# run log: system.lakeflow stand-in recording the bad deploy's window
spark.createDataFrame([
    ("job42", "2026-11-18 13:00", "2026-11-18 14:00", "OK"),
    ("job43", "2026-11-18 14:00", "2026-11-18 15:00", "BAD_DEPLOY"),
    ("job44", "2026-11-18 15:00", "2026-11-18 16:00", "OK"),
], "job string, start string, end string, status string").write.format("delta").mode("overwrite").save(RUNS)

# 1) blast window from the run log
blast = spark.read.format("delta").load(RUNS).where(F.col("status") == "BAD_DEPLOY").collect()
assert len(blast) == 1
lo, hi = int(blast[0]["start"][-5:-3]), int(blast[0]["end"][-5:-3]) - 1
print("blast window from run log: hours", lo, "-", hi)

# 2) confirm damage
before = {r["hour"]: r["revenue"] for r in spark.read.format("delta").load(G11).collect()}
assert before["h14"] == 20.0, before                       # 200 * 0.90 instead of 200 * 0.10
assert before["h13"] == 90.0 and before["h15"] == 270.0 and before["h16"] == 360.0
print("damaged row: h14 =", before["h14"], "(should be 180.0); other hours intact:", before)

# 3) repair: re-transform ONLY the window, MERGE back; capture the pre-repair version first
v_pre = spark.sql(f"DESCRIBE HISTORY delta.`{G11}`").select("version").agg(F.max("version")).collect()[0][0]
fix = transform(0.10, hours=(lo, hi))
fix.createOrReplaceTempView("q11_fix")
spark.sql(f"""
MERGE INTO delta.`{G11}` g USING q11_fix f ON g.hour = f.hour
WHEN MATCHED THEN UPDATE SET g.gross = f.gross, g.revenue = f.revenue
WHEN NOT MATCHED THEN INSERT *
""")

# 4) PROVE the repair
after = {r["hour"]: r["revenue"] for r in spark.read.format("delta").load(G11).collect()}
full = {r["hour"]: r["revenue"] for r in transform(0.10).collect()}
print("after repair:", after)
assert after == full, "repair must equal a full recompute"
old = {r["hour"]: r["revenue"] for r in spark.read.format("delta").option("versionAsOf", v_pre).load(G11).collect()}
changed = [h for h in after if after[h] != old[h]]
assert changed == ["h14"], changed                          # exactly the blast window changed
print("Q11 OK — only h14 corrected (verified via time-travel diff); repaired table == full recompute")
print("""
TALK TRACK — the interview traps:
1) The run log IS the blast-window oracle: system.lakeflow.job_run_timeline gives start/end/status
   per attempt; the first BAD row after a deploy bounds the corrupted window. Version joins
   (DESCRIBE HISTORY) bound it further if jobs write versions.
2) MERGE not OVERWRITE: overwrite rewrites the whole table (new version, full data movement) and
   would have 'repaired' hours that were never broken — and destroyed the audit trail of the bug.
   A scoped MERGE touches only the blast window.
3) PROVE IT twice: equality with a full recompute (correctness) AND a versionAsOf diff (blast
   containment). Interviewers push for exactly this second proof.
4) Comms: the fix is only half the incident — downstream caches, extracts, and finance models
   consumed 20.0 all day. Restatement notice + re-run of downstream consumers closes the loop
   (Day 22's MV rebuild pattern, Day 24's repair-run framing).""")

## Q12 — Data Modeling: Event Sourcing vs State Tables [HARD]

**Interviewer (Revolut):** Ledger balances: do you UPDATE an `accounts` table when money moves, or append immutable events and project state? Build both and make them agree.

**Requirements**

1. An **immutable event log** (deposits, withdrawals — as *negative* deltas — a late-arriving event, and a **correction** modeled as data, never an UPDATE).
2. An **incrementally maintained state table**: MERGE the batch delta of new events into state — selected by the new batch's **event-time window**, NOT a global sequence number (per-account seqs collide: acct2's second event is seq 2).
3. Prove: incrementally maintained state == full log replay == an **as-of** query (balance at any past instant, zero DML).
4. Trade-offs and traps: audit/replay/new projections vs GDPR tombstones and volume; and the **at-least-once trap** — a re-run MERGE double-applies the batch delta, so idempotent consumption needs a stored offset/ack (exactly what checkpointed streaming gives you).

**Sample events** — batch 1: acct1 deposit 500, withdraw −200, acct2 deposit 100; batch 2 (late + correction): acct1 +50, acct2 −30, acct1 −75.

In [0]:
# Solution Q12 — event log + projected state (validated exact notebook flow)
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/etl_test_volume"
try:  # idempotent re-runs: the LOG must start from batch 1 only (append semantics!)
    dbutils.fs.rm(f"{BASE}/d26_q12_events_nb", True)
    dbutils.fs.rm(f"{BASE}/d26_q12_state_nb", True)
except Exception:
    pass
LOG = f"{BASE}/d26_q12_events_nb"; ST = f"{BASE}/d26_q12_state_nb"

# 1) the immutable log — batch 1 arrives
batch1 = spark.createDataFrame([
    ("acct1", 1, "deposit",  500.0, "2026-11-01 09:00"),
    ("acct1", 2, "withdraw", -200.0, "2026-11-02 10:00"),
    ("acct2", 1, "deposit",  100.0, "2026-11-01 11:00"),
], "account string, seq int, event_type string, delta double, ts string")
batch1.write.format("delta").mode("overwrite").save(LOG)

# state initialized from the first batch
batch1.groupBy("account").agg(F.sum("delta").alias("balance"), F.max("ts").alias("last_event_ts")) \
     .write.format("delta").mode("overwrite").save(ST)

# 2) batch 2 arrives LATE and includes a CORRECTION (-75) — appended as data
batch2 = spark.createDataFrame([
    ("acct1", 3, "deposit",    50.0, "2026-11-03 08:00"),
    ("acct2", 2, "withdraw",  -30.0, "2026-11-04 09:00"),
    ("acct1", 4, "correction", -75.0, "2026-11-04 10:00"),
], "account string, seq int, event_type string, delta double, ts string")
batch2.write.format("delta").mode("append").save(LOG)

# 3) incremental maintenance: fold ONLY the new batch's events into state.
#    Selection by EVENT-TIME of the new batch (ts > last processed) — NOT seq > 2:
#    per-account sequences collide (acct2's second event is seq 2 as well).
inc = (spark.read.format("delta").load(LOG).where(F.col("ts") > "2026-11-02 23:59")
       .groupBy("account").agg(F.sum("delta").alias("batch_delta"), F.max("ts").alias("max_ts")))
inc.createOrReplaceTempView("q12_inc")
spark.sql(f"""
MERGE INTO delta.`{ST}` t USING q12_inc s ON t.account = s.account
WHEN MATCHED THEN UPDATE SET t.balance = t.balance + s.batch_delta, t.last_event_ts = s.max_ts
""")

state = {r["account"]: r["balance"] for r in spark.read.format("delta").load(ST).collect()}
replay = {r["account"]: r["balance"] for r in (spark.read.format("delta").load(LOG)
          .groupBy("account").agg(F.sum("delta").alias("balance")).collect())}
as_of = {r["account"]: r["balance"] for r in (spark.read.format("delta").load(LOG)
          .where(F.col("ts") <= "2026-11-03 23:59")
          .groupBy("account").agg(F.sum("delta").alias("balance")).collect())}
print("incremental state:", state)
print("full log replay:   ", replay)
print("as-of 2026-11-03: ", as_of)
assert state == {"acct1": 275.0, "acct2": 70.0}, state
assert replay == state, "projected state must equal a full replay"
assert as_of == {"acct1": 350.0, "acct2": 100.0}, as_of
print("Q12 OK — incrementally maintained state == full log replay; as-of view rebuilt with zero DML")
print("""
TALK TRACK — the interview traps:
1) SUM is an ALGEBRAIC fold: incremental MERGE of batch deltas == full replay (Q6's harness
   logic applied to state). Non-additive state (latest status, max seq event) needs the
   'apply in seq order' pattern instead of a plain sum.
2) SELECTION BUG: 'new events = seq > N' breaks when sequences are PER-ACCOUNT. Select the
   batch by arrival watermark / stored offset — the same rule as streaming checkpoints.
3) AT-LEAST-ONCE TRAP: if the MERGE runs twice on the same batch, deltas apply twice (I hit
   this live during validation: a failed run had already applied -25). Production fix: store
   the last-processed offset IN the state write (same transaction), or use a checkpointed
   streaming source that de-duplicates by design.
4) CORRECTIONS AS DATA: the -75 'reversal fee' is an event, not an UPDATE — the audit trail
   keeps both the original and the correction. GDPR erasure is the cost: tombstone events +
   keyed re-projection (Day 19/25 patterns) instead of DELETE.""")

## Q13 — Data Modeling: The 4-4-5 Fiscal Calendar and Weekly Grain [HARD]

**Interviewer (Walmart):** Retail never reports on raw calendar months — February is 28 days, March is 31, and week-over-week comparisons break across month boundaries. Build the **4-4-5 fiscal calendar**: FY starts Sunday-nearest-Feb-1, 4-week months, 4-4-5 per quarter, 364 days.

**Requirements**

1. Generate the fiscal year programmatically: verify the FY2026 start (2026-02-01) is a Sunday, build a 12-year window, and apply the **53-week catch-up rule** (if next year's start would drift more than 6 days before the Feb-1 anchor, the year is 371 days = 53 weeks) — report which fiscal years are 53-weekers (FY2031, FY2036).
2. Build the week spine: FY / fiscal quarter / fiscal month / fiscal week / week start / week end. Verify: 52 weeks, 13 per quarter, 4-4-5 month pattern, contiguous Sundays.
3. Map a sale date (2026-04-15) to its fiscal week.
4. Explain the reporting pain: the 53rd-week year (week-over-week YoY has a 14-week quarter somewhere), restatement of fixed-month YoY comparisons, and how a `dim_date` exposes fiscal columns alongside calendar ones.

**Sample data** — one FY2026 spine plus one sale date.

In [0]:
# Solution Q13 — 4-4-5 calendar generator (deterministic)
import datetime as dt

anchor_date = dt.date(2026, 2, 1)
assert anchor_date.weekday() == 6, f"FY2026 start must be a Sunday, got {anchor_date.strftime('%A')}"
print("FY2026 starts:", anchor_date, anchor_date.strftime('%A'))

def build_fy(start, n_years=12):
    """52 weeks (364 days) per year; 53 (371) when the next start would drift
    more than 6 days before the Feb-1 anchor — the catch-up week."""
    fys, s = [], start
    for _ in range(n_years):
        nxt = s + dt.timedelta(days=364)             # 52 weeks * 7
        anchor = dt.date(nxt.year, 2, 1)
        weeks = 53 if nxt < anchor - dt.timedelta(days=6) else 52
        fys.append((s.year, weeks))
        s = s + dt.timedelta(days=364 + (7 if weeks == 53 else 0))
    return fys

cal = build_fy(anchor_date)
print("FY (start year, weeks):", cal)
leap53 = [y for y, w in cal if w == 53]
assert all(w in (52, 53) for _, w in cal)
assert leap53 == [2031, 2036], leap53
print("53-week fiscal years in the 12-year window:", leap53)

# month pattern: 4-4-5 per quarter -> 13 weeks/quarter, 52/FY (12 months)
weeks = [w for q in range(1, 5) for w in (4, 4, 5)]
assert len(weeks) == 12 and sum(weeks) == 52
print("month pattern per quarter: [4, 4, 5] -> 13 weeks/quarter, 12 fiscal months, 52 weeks/FY")

# the FY2026 week spine
rows, wk, start = [], 1, anchor_date
for q in range(1, 5):
    mi = 1
    for wcount in (4, 4, 5):                        # fiscal month lengths in weeks
        for _ in range(wcount):
            rows.append((2026, q, mi, wk, start, start + dt.timedelta(days=6)))
            start += dt.timedelta(days=7)
            wk += 1
        mi += 1
assert len(rows) == 52 and rows[-1][5] == dt.date(2027, 1, 30)   # contiguous Sundays
assert all(r[4].weekday() == 6 for r in rows)

sale = dt.date(2026, 4, 15)
hit = [r for r in rows if r[4] <= sale <= r[5]][0]
print(f"sale {sale} -> FY2026 Q{hit[1]} fiscal_month {hit[2]}, fiscal_week {hit[3]} (week of {hit[4]})")
print("Q13 OK — 364-day FY, 4-4-5 quarters (13 wks), 53-week catch-up years:", leap53)
print("""
TALK TRACK — the interview traps:
1) WHY 4-4-5: fixed 13-week quarters make QoQ and WoW comparisons meaningful, and week
   boundaries (Sunday here) never split a promo weekend. Monthly grain = 4 or 5 weeks.
2) THE 53rd WEEK: 364 vs 365.25 -> drift ~1.24 days/yr; when the next start drifts >6 days
   before the anchor, add week 53. FY2031/FY2036 in this window. Comparisons in a 53-week year
   need 'comparable week' logic (week 53 folded, or +1 week offset YoY) — say this out loud,
   it is the #1 retail-reporting gotcha.
3) DIM_DATE: expose calendar AND fiscal columns (date, fy, fiscal_qtr, fiscal_month,
   fiscal_week, is_53_week_year, week_of_year). Reports join the spine — never derive
   fiscal grain in BI tool formulas (five tools, five wrong answers).
4) Sunday-nearest-anchor variants (454, 544) exist; the generator only changes the month
   tuple — mention that flexibility.""")

## Q14 — Snowflake: Budgets and the Cost-Governance Stack [HARD]

**Interviewer (Snowflake SE):** Resource monitors (Days 15/17) bound a warehouse's credits — but leadership wants **monetary budgets in the cloud billing currency**, aggregate across objects, email alerts, and per-team chargeback. Design the whole ladder.

**Requirements**

1. **CREATE BUDGET** syntax: account-level and schema-level budgets, custom frequency (`MONTHLY`), monetary limit, `NOTIFY` at thresholds — write the actual DDL.

2. **MONITOR pattern**: the `SNOWFLAKE.CORE.BUDGETS` / `BUDGET_RESOURCES` views — attach warehouses, databases, schemas; how measured spend accrues daily and what resets.
3. **Notification objects**: email notifications for budgets vs resource monitors' actions (SUSPEND/WAIT); when a budget can *only* alert vs a monitor that can *stop* a warehouse.

4. **The full governance ladder**: warehouse sizing → auto-suspend/resume → resource monitors (credit ceilings, hard stop) → budgets (money ceilings, alert) → tag-based **chargeback** (`ACCOUNT_USAGE` views joined on cost_center tags: cost per query/warehouse/team, per month) — with credit math for a 4-hour daily 2XL workload.

Talk-track + SQL examples.

In [0]:
# Solution Q14 — Budgets + chargeback ladder (talk track; Snowflake-side SQL as strings)
budget_sql = """
-- 1) ACCOUNT-level budget, monthly, in USD (currency of the cloud provider's billing)
CREATE BUDGET IF NOT EXISTS account_monthly
  DATA_TYPE = CURRENCY  AMOUNT = 50000  FREQUENCY = MONTHLY
  START_TIMESTAMP = IMMEDIATE;

-- attach objects: a database, a schema, a warehouse all count toward the SAME budget
ALTER BUDGET account_monthly ADD TO DATABASE analytics_db;
ALTER BUDGET account_monthly ADD TO SCHEMA analytics_db.finance;
ALTER BUDGET account_monthly ADD TO WAREHOUSE team_wh;

-- 2) alert thresholds (% of budget consumed) -> notification integration (email)
CREATE NOTIFICATION INTEGRATION IF NOT EXISTS budget_emails
  TYPE=EMAIL ENABLED=TRUE;
ALTER BUDGET account_monthly SET NOTIFICATION INTEGRATION budget_emails;

-- (a budget ALERTS; it cannot suspend anything. A RESOURCE MONITOR acts:)
CREATE RESOURCE MONITOR wh_ceiling WITH
  CREDIT_QUOTA = 2000  TRIGGERS ON 80 PERCENT DO NOTIFY
                              ON 100 PERCENT DO SUSPEND  ON 110 PERCENT DO SUSPEND_IMMEDIATE;
ALTER WAREHOUSE team_wh SET RESOURCE_MONITOR = wh_ceiling;

-- 3) measured spend lives in SNOWFLAKE.CORE.BUDGET_SCHEMAS / BUDGET_RESOURCES
SELECT * FROM SNOWFLAKE.CORE.BUDGET_RESOURCES
WHERE budget_name = 'ACCOUNT_MONTHLY';   -- schema-level daily accruals

-- 4) chargeback: tags on the objects, spend joined through ACCOUNT_USAGE
WITH wh_spend AS (
  SELECT warehouse_name, SUM(credits_used) AS credits
  FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
  WHERE start_time >= DATE_TRUNC('month', CURRENT_DATE()) GROUP BY 1)
SELECT t.tag_value AS cost_center, s.credits, s.credits * 2.30 AS usd
FROM wh_spend s JOIN SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES t
  ON t.object_name = s.warehouse_name AND t.tag_name = 'cost_center';
"""
print(budget_sql)
# credit math: 2XL = 16 credits/hr * 4 hr/day * 22 workdays
monthly_credits = 16 * 4 * 22
print(f"2XL 4h/day x 22 days = {monthly_credits} credits/month ~= ${monthly_credits * 2.30:,.0f} at $2.30/credit")
assert monthly_credits == 1408
print("Q14 OK (talk track) — monitors stop credits, budgets alert money, tags drive chargeback")
print("""
TALK TRACK — the interview traps:
1) MONITOR vs BUDGET: a resource monitor can SUSPEND a warehouse (hard stop at credit quota);
  a budget only NOTIFIES (money-level visibility across MANY objects). Use monitors for the
  ceiling, budgets for the rollup — they answer different questions.
2) Budgets measure in the cloud provider's currency, accrue DAILY, and reset on the frequency
  boundary (not when you edit the amount — a common misconception).
3) CHARGEBACK needs TAGS: cost_center on warehouses/databases -> ACCOUNT_USAGE joins -> finance
  export. Without tags you have spend, but no owner.
4) The ladder bottom-up: right-size (X-S/L) and auto-suspend FIRST — the cheapest credit is the
  one never burned; monitors/budgets are guardrails, not fixes.""")

## Q15 — Snowflake: Snowflake Scripting — Stored Procedures, Exceptions, Owner's vs Caller's Rights [HARD]

**Interviewer (Goldman Sachs):** Wrap a multi-step load (validate → merge → log) in a **Snowflake Scripting** procedure with exception handling, scheduled by a Task. The procedure must survive partial failures without double-loading.

**Requirements**

1. Snowflake Scripting basics: DECLARE (variables/cursors/RESULTSETs), BEGIN…END, IF/WHILE/RETURN, `EXECUTE IMMEDIATE` for dynamic SQL — how it differs from plain JavaScript UDF-procedures.

2. **Exception handling**: `EXCEPTION WHEN <error> THEN <handler>` with `SQLERRNO`/`SQLERRM`, re-RAISE vs handle-and-continue — write the procedure.

3. **EXECUTE AS OWNER vs CALLER** — the security boundary: whose rights run the DML, who sees the underlying tables, when caller's rights is REQUIRED (logging tables the caller cannot write, source tables the owner should not expose).

4. Wire it into a **Task** (schedule, error integration / event-table logging from Day 19), and deployment via git (Day 25's Snowflake CI/CD).

Talk-track + SQL examples.

In [0]:
# Solution Q15 — Snowflake Scripting procedure (talk track; SQL as strings)
proc_sql = """
CREATE OR REPLACE PROCEDURE analytics_db.finance.load_daily(in_run_date DATE)
RETURNS VARCHAR
LANGUAGE SQL                        -- Snowflake Scripting (not JavaScript)
EXECUTE AS OWNER                    -- runs with the PROCEDURE OWNER's grants, not the caller's
AS
$$
DECLARE
  bad_rows INTEGER;
  merged   INTEGER;
  ex       VARCHAR DEFAULT '';
BEGIN
  -- 1) validate gate
  SELECT COUNT(*) INTO :bad_rows
  FROM staging_orders WHERE order_date = :in_run_date AND amount IS NULL;
  IF (bad_rows > 0) THEN
    RETURN 'BLOCKED: ' || bad_rows || ' null amounts in staging';   -- exit early, no load
  END IF;

  -- 2) merge (idempotent: re-running the same date converges)
  MERGE INTO finance.orders t USING staging_orders s
    ON t.order_id = s.order_id AND t.order_date = :in_run_date
  WHEN MATCHED THEN UPDATE SET t.amount = s.amount
  WHEN NOT MATCHED THEN INSERT VALUES (s.order_id, s.order_date, s.amount);
  merged := SQLROWCOUNT;

  -- 3) structured log write (inside the SAME handler boundary)
  INSERT INTO ops.load_log (run_date, step, detail)
  VALUES (:in_run_date, 'merge', 'rows=' || :merged);

  RETURN 'OK rows=' || merged;

EXCEPTION
  WHEN OTHERS THEN
    -- record and re-raise: the Task's error integration pages on the raised error
    INSERT INTO ops.load_log (run_date, step, detail)
    VALUES (:in_run_date, 'error', SQLERRNO || ': ' || SQLERRM);
    RAISE;                          -- without RAISE you swallow the failure
END;
$$;

-- schedule it: the TASK runs as its owner; errors flow to the event table / error integration
CREATE OR REPLACE TASK analytics_db.finance.t_load_daily
  WAREHOUSE = loader_wh
  SCHEDULE   = 'USING CRON 15 2 * * * America/New_York'
  ERROR_INTEGRATION = ops_error_notif
AS CALL analytics_db.finance.load_daily(CURRENT_DATE() - 1);
ALTER TASK analytics_db.finance.t_load_daily RESUME;
"""
print(proc_sql)
for needed in ["DECLARE", "EXCEPTION", "WHEN OTHERS THEN", "RAISE", "EXECUTE AS OWNER", "SQLROWCOUNT", "ERROR_INTEGRATION"]:
    assert needed in proc_sql, needed
print("Q15 OK (talk track) — validate->merge->log with exception, re-RAISE, task wiring")
print("""
TALK TRACK — the interview traps:
1) OWNER vs CALLER: EXECUTE AS OWNER lets ANALYSTS trigger a load that writes tables they
   CANNOT see — the procedure is the security boundary. CALLER's rights exists for when the
   caller's OWN access must govern (e.g., a utility proc over 'my tables only'). Default for
   CREATE PROCEDURE is CALLER — say that out loud, it surprises people.
2) EXCEPTION ... WHEN OTHERS THEN ... RAISE: handle-log-RAISE, never swallow. SQLERRNO/SQLERRM
   are how the handler identifies the error; RETURN inside an exception ends the call with
   the string, RAISE propagates so the TASK can page.
3) IDempotency: the MERGE key (order_id + run_date) makes re-running the same date a no-op —
   pairs with Day 24's repair-run semantics. Partial failure mid-procedure rolls back the
   statement (not the whole block!) — that is why each step logs.
4) JS vs SQL scripting: Snowflake Scripting (SQL) is now the default; the JS UDF-procedure
   path exists for legacy and for things needing a full language runtime. Event tables
   (Day 19) capture task/procedure telemetry for the Q16-style absence monitoring.""")

## Q16 — Monitoring: Dead-Man's-Switch & Meta-Monitoring — Who Watches the Watchers? [HARD]

**Interviewer (Datadog):** Your scariest failure modes are silent: the pipeline that stopped without erroring, and the alert that never fired because the warehouse was down. Build **alert-on-absence**.

**Requirements**

1. A heartbeat ledger (pipeline, cadence_minutes, last_run_ts) and the staleness rule: **stale = now − last_run > 2× cadence** → page. Flag `stream_enrich` (cadence 30 min, silent for 3 h) while `ingest_orders` (30 min ago) and `nightly_cube` (20 h into a 24 h cadence) stay healthy.
2. **The watchdog's own heartbeat**: the monitoring process must itself emit a heartbeat an independent second path watches (different warehouse, different provider) — meta-monitoring.
3. Synthetic canary consumers and the SQL-alert pattern for "no row in N hours" (`max(event_time) < now() − interval` — an alert on absence, not a threshold).
4. Differentiate from Day 25's tripwires: tripwires watch DATA properties (nulls, volume, dupes); the dead-man's switch watches the WATCHERS (did anything arrive at all?).

In [0]:
# Solution Q16 — alert-on-absence + watchdog self-heartbeat
import datetime as dt

now = dt.datetime(2026, 11, 18, 3, 0)
heartbeats = [
    ("ingest_orders",  60,    now - dt.timedelta(minutes=30)),   # healthy
    ("stream_enrich",  30,    now - dt.timedelta(minutes=180)),  # STALLED: 6x cadence
    ("nightly_cube",   1440,  now - dt.timedelta(hours=20)),     # healthy: 1200 < 2x1440
]
alerts = [p for p, cad, last in heartbeats if (now - last).total_seconds() / 60 > 2 * cad]
print("absence alerts:", alerts)
assert alerts == ["stream_enrich"], alerts

# the watchdog itself is monitored — by a SECOND path (different warehouse/provider)
watchdog = {"watchdog": now - dt.timedelta(minutes=5), "expected_cadence": 5}
watchdog_stale = (now - watchdog["watchdog"]).total_seconds() / 60 > 2 * watchdog["expected_cadence"]
assert not watchdog_stale
print("watchdog heartbeat: 5 min ago -> healthy (the watchdog pages about ITSELF if it stalls)")

sql_alert = """
-- SQL alert on ABSENCE (fires when nothing arrived recently — no threshold on a value)
SELECT (current_timestamp() - MAX(event_time)) AS lag_minutes
FROM streaming_events
HAVING lag_minutes > 120;   -- alert condition: lag_minutes GREATER_THAN 120
"""
print(sql_alert)
print("Q16 OK — alert-on-absence flags stream_enrich; watchdog self-heartbeat verified")
print("""
TALK TRACK — the interview traps:
1) WHY 2x CADENCE: one missed interval = jitter; two = a pattern. Too tight pages on every
   deployment blip; too loose extends outages by the extra timeout.
2) THE SILENT-ALERT FAILURE: if the alert query itself runs on the broken warehouse, it never
   evaluates. The watchdog must live on an INDEPENDENT path — different warehouse, ideally
   different provider — and emit its own heartbeat. Meta-monitoring is the recursive case.
3) TRIPWIRE vs DEAD-MAN (Day 25): a tripwire fires on bad DATA that DID arrive (nulls, volume);
   the dead-man's switch fires on NO data arriving. Both — plus canary consumers (synthetic
   end-to-end writes you can alarm on) — cover the three failure classes.
4) The absence query uses MAX(event_time), never COUNT(*)=0 (count hides WHEN it stopped, and
   an empty partition reads the same as a broken scan).""")

## Q17 — Monitoring: Incident Command & the Blameless Postmortem [FAANG]

**Interviewer (Google SRE):** The 3 AM ladders (Days 16/24) end when the pipeline is green. Staff interviews grade the OTHER half: incident command. Talk me through it, then generate the artifact.

**Requirements**

1. **Severity matrix**: Sev1 (customer-visible / revenue / data-loss risk — IC engaged, exec comms), Sev2 (degraded SLA, internal users — IC if >2h), Sev3 (single pipeline, workaround exists). Who decides, and when does severity CHANGE mid-incident?
2. **Roles**: Incident Commander (coordination, NOT hands-on-keyboard), comms lead, ops/scribe. Why separating the IC from the person typing is non-negotiable.
3. **MTTR** definition (detect → resolve), mitigate vs root-cause, and the comms cadence (e.g., every 30 min).
4. Write the **blameless postmortem**: impact, timeline, root cause as a PROCESS gap (never a person), contributing factors, action items with owners/priorities. Explain why blameless is a systems bet — name-and-shame guarantees you never hear about the next near-miss.

In [0]:
# Solution Q17 — blameless postmortem generator
import datetime as dt

timeline = [
    ("2026-11-18 02:41", " Automated freshness alert fires (gold_orders 2h stale)"),
    ("2026-11-18 02:55", " On-call joins; confirms stream_enrich stalled since 00:00"),
    ("2026-11-18 03:10", " Root cause found: backfill job merged overlapping source rows -> DELTA_MULTIPLE_SOURCE_ROW_CONFLICT, streaming job erroring in retry loop"),
    ("2026-11-18 03:25", " Dedup source view deployed; pipeline resumes"),
    ("2026-11-18 03:32", " Gold fresh; incident closed"),
]
detect, resolve = dt.datetime(2026, 11, 18, 2, 41), dt.datetime(2026, 11, 18, 3, 32)
mttr_min = int((resolve - detect).total_seconds() // 60)

def render_postmortem(timeline, mttr_min):
    md = ["# Postmortem: gold_orders freshness incident (2026-11-18)",
          "**Impact:** 3h stale gold table; 40+ dashboard consumers; 1 SQL alert fired correctly.",
          f"**MTTR:** {mttr_min} minutes (detect -> resolve).",
          "**Root cause (process, not person):** backfill job used an overlapping window "
          "with the live pipeline; MERGE matched multiple source rows per target row.",
          "**Contributing factors:**",
          " 1. No window-overlap check between backfill and live jobs (system gap).",
          " 2. Error surfaced only in driver logs, not as a data-level alert (observability gap).",
          " 3. Retry loop masked the underlying conflict for ~2h (retry design gap).",
          "**Timeline:**"]
    md += [f"  - {ts}:{ev}" for ts, ev in timeline]
    md += ["**Action items:**",
           "  - [P0] Dedupe source in the merge query (owner: data-eng, 1 wk).",
           "  - [P1] Alert when backfill window overlaps live run (owner: platform, 2 wk).",
           "  - [P2] Retry with backoff and circuit-break after N conflicts (owner: data-eng, 2 wk)."]
    return "\n".join(md)

pm = render_postmortem(timeline, mttr_min)
print(pm)
print()
assert "**MTTR:** 51 minutes" in pm, mttr_min
for needed in ["Contributing factors", "Action items", "Timeline", "process, not person"]:
    assert needed in pm, needed
assert mttr_min == 51
print("Q17 OK — blameless postmortem rendered; MTTR = 51 min; sections complete")
print("""
TALK TRACK — the interview traps:
1) IC DOESN'T TYPE: the moment your coordinator is elbows-deep in a fix, NOBODY is tracking
   state, comms, or the next mitigation. Swap roles when the deep-dive starts.
2) MITIGATE FIRST: rollback, freeze, or pin the bad gold table (versionAsOf read-only alias)
   beats a root-cause hunt while customers burn. Root cause can wait for daylight.
3) BLAMELESS IS A SYSTEMS BET: 'the deploy inverted the expression' is a process gap (why did
   CI not catch it? why no gate?); 'Dave inverted the expression' ends the conversation AND
   the near-miss reports forever. Contributing factors, never culprits.
4) ACTION ITEMS have an owner and a date or they are theater. P0 = same week, P1 = same month.""")

## Q18 — Capstone: The 3 AM Multi-System Incident — Synthesis of Days 1–26 [FAANG]

**Interviewer (Netflix, staff loop):** 02:41 — the freshness burn alert fires: `gold_orders` is 2h stale, 40 dashboards degraded. At midnight, a backfill deploy overlapped the live stream's window and duplicated source rows; the streaming MERGE has been erroring in a retry loop ever since. Walk the WHOLE ladder.

**Requirements**

1. **Detect** (Q16 + Day 25): heartbeat absence check — `stream_enrich` silent 3h against a 30-min cadence; DESCRIBE HISTORY freshness evidence on gold.
2. **Diagnose**: reproduce the actual error — MERGE with duplicate source rows raises `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE` (the same live failure as Day 24) — and read the error to its root cause (backfill re-merged an overlapping window).
3. **Repair** (Q11 + Day 24): dedupe the source view (group-by/max), MERGE, verify downstream convergence.
4. **Close** (Q17): the incident ends with the postmortem, not the green dashboard — MTTR 51 minutes, action items P0–P2.

**Sample data** — 3 orders; the backfill re-appends o2 twice and o3 once.

In [0]:
# Solution Q18 — the full ladder, simulated (validated exact notebook flow)
from pyspark.sql import functions as F
import datetime as dt

BASE = "/Volumes/workspace/default/etl_test_volume"
try:  # idempotent re-runs
    dbutils.fs.rm(f"{BASE}/d26_q18_src_nb", True)
    dbutils.fs.rm(f"{BASE}/d26_q18_gold_nb", True)
except Exception:
    pass
SRC = f"{BASE}/d26_q18_src_nb"; G18 = f"{BASE}/d26_q18_gold_nb"

# initial healthy state
spark.createDataFrame([("o1", 100.0), ("o2", 50.0), ("o3", 75.0)], "order_id string, amount double") \
     .write.format("delta").mode("overwrite").save(SRC)
def rebuild_gold():
    (spark.read.format("delta").load(SRC)
     .withColumn("region", F.when(F.col("amount") > 60, "high").otherwise("low"))
     .groupBy("region").agg(F.count("*").alias("orders"), F.sum("amount").alias("total"))
     .write.format("delta").mode("overwrite").save(G18))
rebuild_gold()

# 1) DETECT — heartbeat absence (Q16) + gold freshness evidence (Day 25)
now = dt.datetime(2026, 11, 18, 3, 0)
cadence, silent_for = 30, dt.timedelta(hours=3)
assert (now - (now - silent_for)).total_seconds() / 60 > 2 * cadence
print("detect: stream_enrich silent 3h vs 30-min cadence -> absence alert fires")
last_commit = spark.sql(f"DESCRIBE HISTORY delta.`{G18}`").orderBy(F.col("version").desc()).limit(1).collect()[0]["timestamp"]

def is_stale(last_ts, sla_min=120, ref=None):
    ref = ref or dt.datetime.now()
    return (ref - last_ts.replace(tzinfo=None)).total_seconds() / 60 > sla_min

assert is_stale(now - dt.timedelta(hours=3), 120, ref=now)     # the 00:00 last commit: stale
assert not is_stale(last_commit, 120, ref=dt.datetime.now())  # freshly rebuilt gold: healthy
print("freshness rule verified: a 3h-old commit trips the 120-min SLA; current commit does not")

# 2) DIAGNOSE — the backfill re-appended OVERLAPPING rows; the streaming MERGE conflicts
dup = spark.createDataFrame([("o2", 50.0), ("o2", 50.0), ("o3", 75.0)], "order_id string, amount double")
dup.createOrReplaceTempView("q18_dup")
err = None
try:
    spark.sql(f"""
    MERGE INTO delta.`{SRC}` t USING q18_dup s ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET t.amount = s.amount
    WHEN NOT MATCHED THEN INSERT *
    """)
except Exception as e:
    err = str(e)
assert err is not None and "MULTIPLE SOURCE ROW" in err.upper(), err[:200]
print("diagnose: MERGE raises:", err.splitlines()[0][:110], "...")
print("  -> root cause: backfill window OVERLAPPED the live run; duplicate source rows per key")

# 3) REPAIR — dedupe the source view (same key -> one deterministic row), retry, converge
spark.sql(f"""
MERGE INTO delta.`{SRC}` t
USING (SELECT order_id, max(amount) AS amount FROM q18_dup GROUP BY order_id) s
ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET t.amount = s.amount
WHEN NOT MATCHED THEN INSERT *
""")
rebuild_gold()
final = {r["region"]: (r["orders"], r["total"]) for r in spark.read.format("delta").load(G18).collect()}
print("repair: dedup MERGE succeeds; gold converged:", final)
assert final == {"high": (2, 175.0), "low": (1, 50.0)}, final

# 4) CLOSE — the dashboard going green is NOT the end (Q17): postmortem, MTTR 51 min,
#    P0 dedupe-in-merge, P1 overlap alert, P2 retry-with-backoff.
print("Q18 OK — absence detect -> conflict diagnosis -> dedup repair -> convergence -> postmortem closes it")
print("""
TALK TRACK — walking the ladder in the interview:
1) DETECT (2:41): Day 25's burn-rate freshness tripwire fired; Q16's absence check explains why
   it took 2h (2x cadence rule). Note what DIDN'T fire: the pipeline 'succeeded' — its MERGE
   failed in a RETRY LOOP, so no failure alert ever triggered. Absence, not error, was the signal.
2) DIAGNOSE (2:55-3:10): DESCRIBE HISTORY shows gold's last commit at 00:04 — everything
   downstream is HEALTHY, just frozen. The driver log shows the MERGE error: duplicate source
   rows per target row. Cross-reference the midnight deploy: the backfill window overlapped.
3) REPAIR (3:10-3:25): dedupe the source view (deterministic aggregation per key) -> MERGE
   converges. Q11's version-guard: only the affected rows changed. Consumers re-read the latest
   version; no full rebuild needed.
4) CLOSE (3:32): fresh gold, incident closed — but the STAFF answer is the postmortem: process
   gaps (no overlap check, no data-level alert on the retry loop, masking retries), P0-P2 action
   items. Systems thinking is the whole game at this level.""")

# Day 26 Scorecard

| # | Category | Topic | Tag | Attempted? | Score (0–5) |
|---|----------|-------|-----|-----------|-------------|
| 1 | SQL Advanced | Wilson lower-bound ranking | FAANG | ☐ | |
| 2 | SQL Advanced | Benford's-law fraud screen | FAANG | ☐ | |
| 3 | Python | Producer-consumer + GIL | FAANG | ☐ | |
| 4 | Python | Float error & Kahan summation | FAANG | ☐ | |
| 5 | PySpark | Native `session_window` streaming | FAANG | ☐ | |
| 6 | PySpark | CDF incremental rollup + harness | HARD | ☐ | |
| 7 | Spark Opt | FIFO/FAIR pools & delay scheduling | HARD | ☐ | |
| 8 | Spark Opt | Partitioners & secondary sort | HARD | ☐ | |
| 9 | ETL | Delta Sharing distribution | FAANG | ☐ | |
| 10 | ETL | Lineaged DQ gates | HARD | ☐ | |
| 11 | ETL | Bad-deploy blast-window repair | FAANG | ☐ | |
| 12 | Data Modeling | Event sourcing vs state tables | HARD | ☐ | |
| 13 | Data Modeling | 4-4-5 fiscal calendar | HARD | ☐ | |
| 14 | Snowflake | Budgets & cost governance | HARD | ☐ | |
| 15 | Snowflake | Snowflake Scripting / owner vs caller | HARD | ☐ | |
| 16 | Monitoring | Dead-man's-switch meta-monitoring | HARD | ☐ | |
| 17 | Monitoring | Incident command & postmortem | FAANG | ☐ | |
| 18 | Monitoring | Multi-system incident capstone | FAANG | ☐ | |

**Debrief — the Day 26 through-lines**

* **Correctness proofs as a habit**: the Wilson bound (Q1), the equality harness (Q6), state==replay (Q12), repair==recompute + time-travel containment (Q11/Q18). FAANG staff-loop answers end with "and here is how I'd prove it stayed correct."

* **Absence vs error**: Q16/Q18 — the pipeline that "succeeds" silently, the alert that never evaluates. Monitoring is recursive: something must watch the watcher.

* **Incremental vs rebuild**: algebraic measures (SUM/COUNT) merge from deltas; holistic ones (median) rebuild (Q6); state tables fold from event logs (Q12). Know which class your metric is in before promising freshness.

* **The incident is the job**: Days 16/24/25 built detection and repair; Day 26 closed the loop with command, postmortem, and the synthesis capstone.

**Day 27 candidates (Week 4 continues)**: feature-store point-in-time correctness (leakage prevention — deferred since Day 19), dbt + Databricks orchestration, multi-tenant catalog isolation (catalog-per-tenant vs shared+ABAC), streaming joins across two watermarks (vs Day 19's single-source sessionization), data-product SLOs (TTFB/freshness contracts as APIs).